# NTX 15 — Frozen Granite Regression and Paired Statistics

CPU only. Runs immediately using the real archived Oct 11 case-level predictions. Strictly no fabricated model inference.

## Purpose and provenance

This analyses the **frozen October 11 case-level archive**. It is not a new model run, independent-human dataset, or production exploit fix. The dataset is bundled with SHA-256 verification. Do not pool its 360-case results with the earlier 240-case controlled-only report.

In [ ]:
from pathlib import Path
import sys, os, csv, json, hashlib, zipfile, subprocess, time
import pandas as pd
from datetime import datetime, timezone
KIT = Path('/content/NiyamTraceX_Paper_Final_Validation_Kit') if Path('/content').exists() else Path.cwd().parent
if not (KIT/'evidence').is_dir():
    here = Path.cwd()
    choices = [here, here.parent, Path('/mnt/data/NiyamTraceX_Paper_Final_Validation_Kit')]
    for candidate in choices:
        if (candidate/'evidence').is_dir(): KIT = candidate; break
if not (KIT/'evidence').is_dir():
    package = None
    for p in [Path.cwd()/'NiyamTraceX_Paper_Final_Validation_Kit.zip',Path('/content/NiyamTraceX_Paper_Final_Validation_Kit.zip')]:
        if p.exists(): package=p;break
    if package is None and 'google.colab' in sys.modules:
        from google.colab import files
        print('Upload NiyamTraceX_Paper_Final_Validation_Kit.zip (the full kit).')
        upload = files.upload()
        if 'NiyamTraceX_Paper_Final_Validation_Kit.zip' in upload:
            package = Path.cwd()/'NiyamTraceX_Paper_Final_Validation_Kit.zip'
    if package:
        with zipfile.ZipFile(package) as z:
            for part in z.infolist():
                target=(Path('/content') if Path('/content').exists() else Path.cwd())/part.filename
                if not str(target.resolve()).startswith(str((Path('/content') if Path('/content').exists() else Path.cwd()).resolve())):
                    raise RuntimeError('Unsafe archive path')
            z.extractall('/content' if Path('/content').exists() else '.')
        KIT = (Path('/content') if Path('/content').exists() else Path.cwd())/'NiyamTraceX_Paper_Final_Validation_Kit'
OUT = Path(os.getenv('NTX_PAPER_OUT',str(KIT/'outputs')))
OUT.mkdir(parents=True,exist_ok=True)
print('Kit:',KIT,'\nOutput:',OUT)
assert (KIT/'evidence').is_dir(), 'Evidence not found. Upload the full kit ZIP.'
def dump(obj,p):
    Path(p).parent.mkdir(parents=True,exist_ok=True)
    Path(p).write_text(json.dumps(obj,indent=2,default=str),encoding='utf-8')
def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

In [ ]:
P=KIT/'evidence'/'paired_variant_cases.csv'
meta=json.loads((KIT/'evidence'/'EVIDENCE_PROVENANCE.json').read_text())
assert sha(P)==meta['sha256'], 'Source data altered'
df=pd.read_csv(P,keep_default_na=False)
need={'direction','id','group','language','kind','expected','variant','decision','safe_completion','unsafe_case','unauthorized_effects','reason'}
assert need.issubset(df.columns), sorted(need-set(df.columns))
assert len(df)==4320, f'Unexpected {len(df)} rows'
assert not df.duplicated(['direction','id','variant']).any()
assert set(df.direction)=={'granite_primary','qwen_primary'}
assert set(df.variant)=={'raw','policy_only','anchors','full','single_best','independent_only'},set(df.variant)
for (direction,variant),g in df.groupby(['direction','variant']):
    assert len(g)==360 and g.id.nunique()==360,(direction,variant)
    assert set(g.id)==set(df[(df.direction==direction)&(df.variant=='anchors')].id)
for f in ['safe_completion','unsafe_case']:
    df[f]=df[f].astype(str).str.lower().map({'true':True,'false':False})
    assert df[f].notna().all()
df['correct']=df['decision']==df['expected']
df['unauthorized_effects']=pd.to_numeric(df['unauthorized_effects'],errors='raise')
print('Verified raw cases:',len(df),'source SHA256:',meta['sha256'])

In [ ]:
from scipy.stats import binomtest
from statsmodels.stats.proportion import proportion_confint
rows=[]
for (model,variant),g in df.groupby(['direction','variant']):
    a=g[g.expected=='ALLOW']
    rows.append(dict(direction=model,variant=variant,n=len(g),correct=int(g.correct.sum()),accuracy=float(g.correct.mean()),
        allow_n=len(a),safe_completion_count=int(a.safe_completion.sum()),safe_completion_rate=float(a.safe_completion.mean()) if len(a) else None,
        unsafe_n=int(g.unsafe_case.sum()),unauthorized_rows=int(g.unauthorized_effects.sum()),
        unsafe_upper_95=float(proportion_confint(int(g.unsafe_case.sum()),len(g),alpha=.05,method='beta')[1])))
summary=pd.DataFrame(rows).sort_values(['direction','variant'])
summary.to_csv(OUT/'15_full_cohort_summary.csv',index=False)
print(summary[['direction','variant','n','accuracy','safe_completion_rate','unsafe_n','unauthorized_rows','unsafe_upper_95']].to_string(index=False))

In [ ]:
paired=[]; bad=[]
for direction in sorted(df.direction.unique()):
    d=df[df.direction==direction]
    a=d[d.variant=='anchors'].set_index('id').sort_index()
    f=d[d.variant=='full'].set_index('id').sort_index()
    assert a.index.equals(f.index)
    diff=a.decision!=f.decision
    b=int((a.correct & ~f.correct).sum());c=int((~a.correct & f.correct).sum())
    mc=binomtest(min(b,c), n=b+c, p=.5,alternative='two-sided').pvalue if b+c else 1.0
    paired.append(dict(direction=direction,n=len(a),divergent_decisions=int(diff.sum()),anchor_only_correct=b,full_only_correct=c,exact_mcnemar_p=mc,
                       anchor_accuracy=float(a.correct.mean()),full_accuracy=float(f.correct.mean()),delta_full_minus_anchor=float(f.correct.mean()-a.correct.mean())))
    comp=pd.DataFrame({'id':a.index,'group':a.group.values,'language':a.language.values,'kind':a.kind.values,'expected':a.expected.values,
      'anchor':a.decision.values,'full':f.decision.values,'anchor_correct':a.correct.values,'full_correct':f.correct.values,
      'anchor_reason':a.reason.values,'full_reason':f.reason.values,'anchor_safe_completion':a.safe_completion.values,'full_safe_completion':f.safe_completion.values})
    bad.append(comp.loc[diff.to_numpy()].assign(direction=direction))
paired=pd.DataFrame(paired)
regressions=pd.concat(bad,ignore_index=True)
paired.to_csv(OUT/'15_paired_significance.csv',index=False)
regressions.to_csv(OUT/'15_regression_case_ids.csv',index=False)
assert len(regressions.query("direction == 'granite_primary'"))==48
assert len(regressions.query("direction == 'qwen_primary'"))==0
print(paired.to_string(index=False)); print('\nGranite failure classes:');print(regressions.groupby(['direction','kind','full_reason']).size())

In [ ]:
# Cluster bootstrap uses semantic group, rather than falsely treating translations as independent.
import numpy as np
rng=np.random.default_rng(20261011)
boot=[]
for direction in sorted(df.direction.unique()):
    a=df.query('direction == @direction and variant == "anchors"').set_index('id').sort_index()
    f=df.query('direction == @direction and variant == "full"').set_index('id').sort_index()
    v=pd.DataFrame({'group':a.group,'delta':f.correct.astype(int)-a.correct.astype(int)})
    means=v.groupby('group')['delta'].mean().to_numpy()
    draws=rng.choice(means,size=(5000,len(means)),replace=True).mean(axis=1)
    boot.append({'direction':direction,'n_groups':len(means),'delta_point':float(v.delta.mean()),'cluster_bootstrap_low':float(np.quantile(draws,.025)),'cluster_bootstrap_high':float(np.quantile(draws,.975))})
bt=pd.DataFrame(boot);bt.to_csv(OUT/'15_cluster_bootstrap.csv',index=False)
print(bt.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(7,4))
models=['qwen_primary','granite_primary']
x=np.arange(len(models));w=.32
s=summary.set_index(['direction','variant'])
ax.bar(x-w/2,[s.loc[(m,'anchors'),'accuracy'] for m in models],width=w,label='Anchor Lock')
ax.bar(x+w/2,[s.loc[(m,'full'),'accuracy'] for m in models],width=w,label='Full intersection')
ax.set_xticks(x,models);ax.set_ylim(0,1.04);ax.set_ylabel('Decision accuracy (360 cases/model)');ax.legend();fig.tight_layout()
fig.savefig(OUT/'15_accuracy_paired.png',dpi=180);plt.show()
report={'source_archive':meta['source_archive'],'source_sha256':meta['sha256'],'n_case_rows':len(df),'paired':paired.to_dict('records'),
  'cluster_bootstrap':bt.to_dict('records'),'granite_regression_count':48,'cannot_claim':['independent candidate superiority','production safety','human-labeled multilingual generalization'],
  'conclusion':'Full method does not outperform anchors in October 11 cohort; Granite degrades. Regressions require investigation rather than exclusion.'}
dump(report,OUT/'15_AUDIT_MANIFEST.json')
print('Saved:',[p.name for p in OUT.glob('15_*')])